# Notebook 02 — Finalize Bible Reference Abbreviations in Revelation SVG Outputs

This notebook performs deterministic final cleanup on already-translated, already-collapsed SVG files.

It does not call an LLM and does not translate anything. It loads the full prepared unit state from Notebook 00 to identify Bible-reference-related elements, then loads the already-collapsed SVG files from Notebook 01 and modifies only selected text content in those SVGs.

The collapsed SVG files are the artifacts modified here. This notebook does not reconstruct SVGs from scratch from JSON. The JSON/dataframe is used only as a map to identify which SVG elements are Bible-reference-related.

This notebook writes new final SVG files and does not overwrite previous outputs.

In [1]:
from pathlib import Path
from datetime import datetime
from collections import Counter
import json
import re

import pandas as pd
from lxml import etree

PROJECT_ROOT = Path.cwd()
JSON_DIR = PROJECT_ROOT / "json_files"
SVG_OUTPUT_DIR = PROJECT_ROOT / "svg_output_files"

FULL_UNITS_PATH = JSON_DIR / "translation_units_full_prepared.json"
TRANSLATIONS_PATH = JSON_DIR / "translations_Spanish_20260520_2017.json"

COLLAPSED_SVG_FILES = {
    # "SevenChurchesOfRevelation.svg": SVG_OUTPUT_DIR / "SevenChurchesOfRevelation_spanish_20260520_1459_collapsed_tspans.svg",
    "StructureOfRevelation.svg": SVG_OUTPUT_DIR / "StructureOfRevelation_spanish_20260520_2017_collapsed_tspans.svg",
}


def workflow_relative_path(path: Path) -> Path:
    try:
        return path.relative_to(PROJECT_ROOT.parent)
    except ValueError:
        return path


def infer_target_language_from_translations(path: Path):
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        for key in ["target_language", "language", "targetLanguage", "target"]:
            value = data.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip(), f"translation JSON metadata key: {key}"

        metadata = data.get("metadata")
        if isinstance(metadata, dict):
            for key in ["target_language", "language", "targetLanguage", "target"]:
                value = metadata.get(key)
                if isinstance(value, str) and value.strip():
                    return value.strip(), f"translation JSON metadata key: metadata.{key}"

    filename_match = re.search(r"translations_([A-Za-z]+)_\d{8}_\d{4}\.json$", path.name)
    if filename_match:
        return filename_match.group(1), "translation JSON filename"

    raise ValueError(
        "Could not infer target language from translation JSON metadata or filename: "
        f"{path.name}"
    )


target_language, target_language_source = infer_target_language_from_translations(TRANSLATIONS_PATH)
target_language_slug = target_language.strip().lower().replace(" ", "_")
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print("PROJECT_ROOT:", workflow_relative_path(PROJECT_ROOT))
print("FULL_UNITS_PATH:", workflow_relative_path(FULL_UNITS_PATH))
print("TRANSLATIONS_PATH:", workflow_relative_path(TRANSLATIONS_PATH))
print("SVG_OUTPUT_DIR:", workflow_relative_path(SVG_OUTPUT_DIR))
print("target_language:", target_language)
print("target_language_source:", target_language_source)
print("target_language_slug:", target_language_slug)
print("timestamp:", timestamp)

PROJECT_ROOT: rev
FULL_UNITS_PATH: rev\json_files\translation_units_full_prepared.json
TRANSLATIONS_PATH: rev\json_files\translations_Spanish_20260520_2017.json
SVG_OUTPUT_DIR: rev\svg_output_files
target_language: Spanish
target_language_source: translation JSON filename
target_language_slug: spanish
timestamp: 20260520_2024


In [2]:
def load_json_records(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return pd.DataFrame(data)

    if isinstance(data, dict):
        if isinstance(data.get("units"), list):
            return pd.DataFrame(data["units"])

        list_keys = [k for k, v in data.items() if isinstance(v, list)]
        if len(list_keys) == 1:
            return pd.DataFrame(data[list_keys[0]])

        obvious_keys = ["records", "rows", "data", "translations", "results"]
        for key in obvious_keys:
            if isinstance(data.get(key), list):
                return pd.DataFrame(data[key])

    raise ValueError(f"Could not identify records in JSON file: {workflow_relative_path(path)}")


df_units_full = load_json_records(FULL_UNITS_PATH)
df_translations = load_json_records(TRANSLATIONS_PATH)

print("Full unit rows:", len(df_units_full))
print("Translation result rows:", len(df_translations))
print("Full unit columns:", list(df_units_full.columns))
print("Translation columns:", list(df_translations.columns))

if "translation_action" in df_units_full.columns:
    print("\ntranslation_action counts:")
    print(df_units_full["translation_action"].value_counts(dropna=False))

if "bible_reference_category" in df_units_full.columns:
    print("\nbible_reference_category counts:")
    print(df_units_full["bible_reference_category"].value_counts(dropna=False))

if "source_file" in df_units_full.columns:
    print("\nsource_file counts:")
    print(df_units_full["source_file"].value_counts(dropna=False))

display(df_units_full.head())
display(df_translations.head())

Full unit rows: 249
Translation result rows: 164
Full unit columns: ['unit_key', 'unit_type', 'source_file', 'group_stack', 'element_path', 'text_id', 'tspan_id', 'tspan_idx', 'source_text', 'translation_action', 'translation_note', 'target_text', 'contains_bible_reference', 'bible_reference_category', 'bible_reference_note', 'repeated_phrase_note']
Translation columns: ['unit_key', 'translated_text']

translation_action counts:
translation_action
translate    164
preserve      85
Name: count, dtype: int64

bible_reference_category counts:
bible_reference_category
None                    167
cross_reference_only     43
reference_only           31
reference_with_text       8
Name: count, dtype: int64

source_file counts:
source_file
StructureOfRevelation.svg    249
Name: count, dtype: int64


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,translation_action,translation_note,target_text,contains_bible_reference,bible_reference_category,bible_reference_note,repeated_phrase_note
0,65971727bf7e86f3fecbb33b07be3acc2892de35,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[10],None,None,None,cf. Ex 9:9-11,preserve,Bible cross-reference preserved unchanged.,cf. Ex 9:9-11,True,cross_reference_only,Cross-reference beginning with cf. detected.,
1,8140f9ef3ce81a748de1998b176b8d1c1c072cd5,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[11],None,None,None,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",preserve,Bible cross-reference preserved unchanged.,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",True,cross_reference_only,Cross-reference beginning with cf. detected.,
2,69c90ab68fd7372766dc12d6f01b35d3857127b6,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[12],None,None,None,cf. Rev 14:18,preserve,Bible cross-reference preserved unchanged.,cf. Rev 14:18,True,cross_reference_only,Cross-reference beginning with cf. detected.,
3,3bf7cce880b6e43304712659ba23c60caf957731,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[13],None,None,None,"cf. Rev 9:14,13:1",preserve,Bible cross-reference preserved unchanged.,"cf. Rev 9:14,13:1",True,cross_reference_only,Cross-reference beginning with cf. detected.,
4,4041ecaccb42dee3d7f29c75e00b137f86bf10cc,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[14],None,None,None,232 cross-references,translate,,None,False,None,,"Repeated phrase detected: ""cross-reference(s)""..."


,unit_key,translated_text
0,4041ecaccb42dee3d7f29c75e00b137f86bf10cc,232 referencias cruzadas
1,6bb999b7f7c222c530baaa3de7036f1e146d9037,Toda vida en el mar muere
2,2e46ca8ff13d19e17a83d5683f948b0b18842b2d,Agua en sangre
3,fc6ae5198a7f6b72c6c870f6aae0f115be396976,Tinieblas
4,7fb015fbd2ea5ae5e874ec080f2ca02f6d409f96,7 Copas


In [3]:
REFERENCE_CATEGORIES = {
    "cross_reference_only",
    "reference_only",
    "leading_reference_with_text",
    "reference_with_text",
}

df_reference_units = df_units_full[
    df_units_full["bible_reference_category"].isin(REFERENCE_CATEGORIES)
].copy()

print("Total reference units:", len(df_reference_units))
print("\nCounts by bible_reference_category:")
print(df_reference_units["bible_reference_category"].value_counts(dropna=False))
print("\nCounts by translation_action:")
print(df_reference_units["translation_action"].value_counts(dropna=False))
print("\nCounts by source_file:")
print(df_reference_units["source_file"].value_counts(dropna=False))

reference_review_cols = [
    "source_file",
    "source_text",
    "translation_action",
    "bible_reference_category",
    "element_path",
    "target_text",
]
display(df_reference_units[reference_review_cols])

Total reference units: 82

Counts by bible_reference_category:
bible_reference_category
cross_reference_only    43
reference_only          31
reference_with_text      8
Name: count, dtype: int64

Counts by translation_action:
translation_action
preserve     74
translate     8
Name: count, dtype: int64

Counts by source_file:
source_file
StructureOfRevelation.svg    82
Name: count, dtype: int64


,source_file,source_text,translation_action,bible_reference_category,element_path,target_text
0,StructureOfRevelation.svg,cf. Ex 9:9-11,preserve,cross_reference_only,svg/g#_x37__bowls/text[10],cf. Ex 9:9-11
1,StructureOfRevelation.svg,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6",preserve,cross_reference_only,svg/g#_x37__bowls/text[11],"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6"
2,StructureOfRevelation.svg,cf. Rev 14:18,preserve,cross_reference_only,svg/g#_x37__bowls/text[12],cf. Rev 14:18
3,StructureOfRevelation.svg,"cf. Rev 9:14,13:1",preserve,cross_reference_only,svg/g#_x37__bowls/text[13],"cf. Rev 9:14,13:1"
6,StructureOfRevelation.svg,Rev 16:3,preserve,reference_only,svg/g#_x37__bowls/text[16],Rev 16:3
...,...,...,...,...,...,...
239,StructureOfRevelation.svg,"cf. Isa 33; Jer 1Eze 1-3,7-12,14,37,40Dan 7,12...",preserve,cross_reference_only,svg/g#interlude/text[7],"cf. Isa 33; Jer 1Eze 1-3,7-12,14,37,40Dan 7,12..."
240,StructureOfRevelation.svg,"cf. Dan 7.2, Eze 9:4",preserve,cross_reference_only,svg/g#interlude/text[8],"cf. Dan 7.2, Eze 9:4"
241,StructureOfRevelation.svg,cf. Eze 48,preserve,cross_reference_only,svg/g#interlude/text[9],cf. Eze 48
244,StructureOfRevelation.svg,After these things I looked and saw a door ope...,translate,reference_with_text,svg/g#throne_room/text[2],None


In [4]:
def localname(tag):
    if not isinstance(tag, str):
        return ""
    return tag.split("}", 1)[-1] if tag.startswith("{") else tag


def parse_svg(svg_path: Path):
    parser = etree.XMLParser(recover=False, huge_tree=True, remove_blank_text=False)
    return etree.parse(str(svg_path), parser)


def _parse_path_part(part: str):
    m = re.fullmatch(r"([A-Za-z0-9_:-]+)(?:#(.+)|\[(\d+)\])?", part)
    if not m:
        return None
    tag, element_id, idx = m.groups()
    return tag, element_id, int(idx) if idx else None


def _direct_children_matching_tag(parent, tag_name):
    return [child for child in parent if localname(child.tag) == tag_name]


def find_by_element_path(root, path):
    if not isinstance(path, str) or not path.strip():
        return None

    parts = [p for p in path.strip().split("/") if p]
    if not parts:
        return None

    first = _parse_path_part(parts[0])
    if not first or first[0] != localname(root.tag):
        return None

    cur = root
    for part in parts[1:]:
        parsed = _parse_path_part(part)
        if not parsed:
            return None
        tag, element_id, idx = parsed

        candidates = _direct_children_matching_tag(cur, tag)
        if element_id is not None:
            candidates = [el for el in candidates if el.get("id") == element_id]
        if idx is not None:
            idx0 = idx - 1
            if idx0 < 0 or idx0 >= len(candidates):
                return None
            cur = candidates[idx0]
        else:
            if len(candidates) != 1:
                return None
            cur = candidates[0]

    return cur


def _find_descendant_by_tag_and_id(parent, tag_name, element_id):
    for el in parent.iter():
        if el is parent:
            continue
        if localname(el.tag) == tag_name and el.get("id") == element_id:
            return el
    return None


def _resolve_direct_indexed_child(parent, path_part):
    parsed = _parse_path_part(path_part)
    if not parsed:
        return None
    tag, element_id, idx = parsed
    candidates = _direct_children_matching_tag(parent, tag)
    if element_id is not None:
        candidates = [el for el in candidates if el.get("id") == element_id]
    if idx is not None:
        idx0 = idx - 1
        if idx0 < 0 or idx0 >= len(candidates):
            return None
        return candidates[idx0]
    if len(candidates) == 1:
        return candidates[0]
    return None


def find_by_element_path_with_id_fallback(root, path):
    direct = find_by_element_path(root, path)
    if direct is not None:
        return direct

    if not isinstance(path, str) or not path.strip():
        return None

    parts = [p for p in path.strip().split("/") if p]
    parsed_parts = [(part, _parse_path_part(part)) for part in parts]
    if any(parsed is None for _, parsed in parsed_parts):
        return None

    id_positions = [i for i, (_, parsed) in enumerate(parsed_parts) if parsed[1] is not None]
    if not id_positions:
        return None

    current = None
    last_id_pos = None
    for n, pos in enumerate(id_positions):
        _, (tag, element_id, _idx) = parsed_parts[pos]
        if n == 0:
            current = _find_descendant_by_tag_and_id(root, tag, element_id)
        else:
            current = _find_descendant_by_tag_and_id(current, tag, element_id)
        if current is None:
            return None
        last_id_pos = pos

    for part, _parsed in parsed_parts[last_id_pos + 1:]:
        current = _resolve_direct_indexed_child(current, part)
        if current is None:
            return None

    return current


def _direct_tspans(el):
    return [child for child in el if localname(child.tag) == "tspan"]


def get_visible_text(el):
    if el is None:
        return ""
    return "".join(el.itertext())


def set_visible_text_conservative(el, new_text):
    tspans = _direct_tspans(el)
    if tspans:
        el.text = el.text if el.text is not None else None
        tspans[0].text = new_text
        for extra in tspans[1:]:
            extra.text = ""
            if extra.tail is not None:
                extra.tail = ""
    else:
        el.text = new_text

In [5]:
# Notebook-local house-style replacement map.
# bible_reference_helpers.py provides book/alias metadata, but this dictionary
# defines the specific Spanish abbreviation normalization policy for this SVG output.
# Keep this explicit and reviewable for the first pass.

import bible_reference_helpers as brh

REFERENCE_ALIAS_REPLACEMENTS_ES = {
    "Rev": "Ap",
    "Revelation": "Apocalipsis",
    "Ezek": "Ez",
    "Eze": "Ez",
    "Josh": "Jos",
    "Judg": "Jue",
    "Jdg": "Jue",
    "Acts": "Hch",
    "Luke": "Lc",
    "Lk": "Lc",
    "Ps": "Sal",
    "Pss": "Sal",
    "1 Kgs": "1 R",
    "2 Kgs": "2 R",
    "1 Ki": "1 R",
    "2 Ki": "2 R",
}


def normalize_bible_reference_abbreviations(text, replacements):
    if text is None:
        return text, []

    text = str(text)
    if not text:
        return text, []

    aliases = sorted(replacements.keys(), key=len, reverse=True)
    alias_pattern = "|".join(re.escape(alias) for alias in aliases)

    pattern = re.compile(
        rf"(?<![A-Za-z])(?P<alias>{alias_pattern})(?P<space>\s+)"
        rf"(?P<chapter>\d+)"
        rf"(?P<verse_part>(?::\d+(?:[-–]\d+)?)?)"
        rf"(?P<range_part>(?:\s*[-–]\s*\d+(?::\d+(?:[-–]\d+)?)?)?)"
        rf"(?P<trailing>[*.,;:]?)",
        flags=re.IGNORECASE,
    )

    replacements_made = []

    def replace_match(m):
        source_alias = m.group("alias")
        replacement_alias = None
        for key, value in replacements.items():
            if key.lower() == source_alias.lower():
                replacement_alias = value
                break
        if replacement_alias is None:
            return m.group(0)

        before_match = m.group(0)
        after_match = (
            replacement_alias
            + m.group("space")
            + m.group("chapter")
            + m.group("verse_part")
            + m.group("range_part")
            + m.group("trailing")
        )

        replacements_made.append(
            {
                "source_alias": source_alias,
                "replacement_alias": replacement_alias,
                "before_match": before_match,
                "after_match": after_match,
            }
        )
        return after_match

    normalized_text = pattern.sub(replace_match, text)
    return normalized_text, replacements_made

In [6]:
examples = [
    "cf. Jer 2:2, Gen 2:9, Rev 22:2",
    "Rev 14:6-13*",
    "Rev 13*",
    "Revelation 21-22",
    "Revelation 10 – 11:14",
    "There were lightnings, sounds, thunders; there was a great earthquake... Rev 16:18",
    "cf. 1 Ki 22,Is 6,Jer 17:12,Eze 1,3:12-14,Eze 10",
    "Numbers in Revelation",
]

for example in examples:
    normalized, details = normalize_bible_reference_abbreviations(
        example,
        REFERENCE_ALIAS_REPLACEMENTS_ES,
    )
    print("BEFORE:", example)
    print("AFTER: ", normalized)
    print("DETAILS:", details)
    print()

BEFORE: cf. Jer 2:2, Gen 2:9, Rev 22:2
AFTER:  cf. Jer 2:2, Gen 2:9, Ap 22:2
DETAILS: [{'source_alias': 'Rev', 'replacement_alias': 'Ap', 'before_match': 'Rev 22:2', 'after_match': 'Ap 22:2'}]

BEFORE: Rev 14:6-13*
AFTER:  Ap 14:6-13*
DETAILS: [{'source_alias': 'Rev', 'replacement_alias': 'Ap', 'before_match': 'Rev 14:6-13*', 'after_match': 'Ap 14:6-13*'}]

BEFORE: Rev 13*
AFTER:  Ap 13*
DETAILS: [{'source_alias': 'Rev', 'replacement_alias': 'Ap', 'before_match': 'Rev 13*', 'after_match': 'Ap 13*'}]

BEFORE: Revelation 21-22
AFTER:  Apocalipsis 21-22
DETAILS: [{'source_alias': 'Revelation', 'replacement_alias': 'Apocalipsis', 'before_match': 'Revelation 21-22', 'after_match': 'Apocalipsis 21-22'}]

BEFORE: Revelation 10 – 11:14
AFTER:  Apocalipsis 10 – 11:14
DETAILS: [{'source_alias': 'Revelation', 'replacement_alias': 'Apocalipsis', 'before_match': 'Revelation 10 – 11:14', 'after_match': 'Apocalipsis 10 – 11:14'}]

BEFORE: There were lightnings, sounds, thunders; there was a great ear

In [7]:
preview_rows = []
unresolved_rows = []
inspected_count = 0
found_count = 0

for source_file, collapsed_svg_path in COLLAPSED_SVG_FILES.items():
    units_for_file = df_reference_units[df_reference_units["source_file"].eq(source_file)]
    tree = parse_svg(collapsed_svg_path)
    root = tree.getroot()

    for _, row in units_for_file.iterrows():
        inspected_count += 1
        el = find_by_element_path_with_id_fallback(root, row["element_path"])
        if el is None:
            unresolved_rows.append(row.to_dict())
            continue

        found_count += 1
        current_text = get_visible_text(el)
        normalized_text, replacements_made = normalize_bible_reference_abbreviations(
            current_text,
            REFERENCE_ALIAS_REPLACEMENTS_ES,
        )

        if normalized_text != current_text:
            preview_rows.append(
                {
                    "source_file": source_file,
                    "source_text": row["source_text"],
                    "current_svg_text": current_text,
                    "normalized_svg_text": normalized_text,
                    "bible_reference_category": row["bible_reference_category"],
                    "translation_action": row["translation_action"],
                    "element_path": row["element_path"],
                    "replacement_count": len(replacements_made),
                    "replacement_details": replacements_made,
                }
            )

df_reference_normalization_preview = pd.DataFrame(preview_rows)
df_unresolved_reference_rows = pd.DataFrame(unresolved_rows)

replacement_counter = Counter()
for details in df_reference_normalization_preview.get("replacement_details", []):
    for item in details:
        replacement_counter[(item["source_alias"], item["replacement_alias"])] += 1

print("Reference rows inspected:", inspected_count)
print("Elements found:", found_count)
print("Unresolved paths:", len(df_unresolved_reference_rows))
print("Rows that would change:", len(df_reference_normalization_preview))
print("\nReplacement counts by source_alias -> replacement_alias:")
for (source_alias, replacement_alias), count in replacement_counter.most_common():
    print(f"{source_alias} -> {replacement_alias}: {count}")

display(df_reference_normalization_preview)
if not df_unresolved_reference_rows.empty:
    display(df_unresolved_reference_rows[["source_file", "source_text", "element_path", "bible_reference_category"]])

Reference rows inspected: 82
Elements found: 82
Unresolved paths: 0
Rows that would change: 56

Replacement counts by source_alias -> replacement_alias:
Rev -> Ap: 43
Eze -> Ez: 15
Ps -> Sal: 8
1 Ki -> 1 R: 1


,source_file,source_text,current_svg_text,normalized_svg_text,bible_reference_category,translation_action,element_path,replacement_count,replacement_details
0,StructureOfRevelation.svg,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6","cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6","cf. Ex 7:17-21,Is 49:26,Ap 8:8-10,11:6",cross_reference_only,preserve,svg/g#_x37__bowls/text[11],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."
1,StructureOfRevelation.svg,cf. Rev 14:18,cf. Rev 14:18,cf. Ap 14:18,cross_reference_only,preserve,svg/g#_x37__bowls/text[12],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."
2,StructureOfRevelation.svg,"cf. Rev 9:14,13:1","cf. Rev 9:14,13:1","cf. Ap 9:14,13:1",cross_reference_only,preserve,svg/g#_x37__bowls/text[13],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."
3,StructureOfRevelation.svg,Rev 16:3,Rev 16:3,Ap 16:3,reference_only,preserve,svg/g#_x37__bowls/text[16],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."
4,StructureOfRevelation.svg,Rev 16:4-7,Rev 16:4-7,Ap 16:4-7,reference_only,preserve,svg/g#_x37__bowls/text[18],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."
5,StructureOfRevelation.svg,Rev 16:10-11,Rev 16:10-11,Ap 16:10-11,reference_only,preserve,svg/g#_x37__bowls/text[20],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."
6,StructureOfRevelation.svg,"cf. Joe 2,Rev 6:12","cf. Joe 2,Rev 6:12","cf. Joe 2,Ap 6:12",cross_reference_only,preserve,svg/g#_x37__bowls/text[6],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."
7,StructureOfRevelation.svg,Rev 16:2,Rev 16:2,Ap 16:2,reference_only,preserve,svg/g#_x37__bowls/text[7],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."
8,StructureOfRevelation.svg,Rev 16:8-9,Rev 16:8-9,Ap 16:8-9,reference_only,preserve,svg/g#_x37__bowls/text[8],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."
9,StructureOfRevelation.svg,Rev 16:12-14,Rev 16:12-14,Ap 16:12-14,reference_only,preserve,svg/g#_x37__bowls/text[9],1,"[{'source_alias': 'Rev', 'replacement_alias': ..."


In [8]:
final_svg_outputs = []
actual_change_rows = []

for source_file, collapsed_svg_path in COLLAPSED_SVG_FILES.items():
    units_for_file = df_reference_units[df_reference_units["source_file"].eq(source_file)]
    tree = parse_svg(collapsed_svg_path)
    root = tree.getroot()

    inspected = 0
    found = 0
    changed = 0
    unresolved = 0
    replacement_count = 0

    for _, row in units_for_file.iterrows():
        inspected += 1
        el = find_by_element_path_with_id_fallback(root, row["element_path"])
        if el is None:
            unresolved += 1
            continue

        found += 1
        old_text = get_visible_text(el)
        new_text, replacements_made = normalize_bible_reference_abbreviations(
            old_text,
            REFERENCE_ALIAS_REPLACEMENTS_ES,
        )

        if new_text != old_text:
            set_visible_text_conservative(el, new_text)
            changed += 1
            replacement_count += len(replacements_made)
            actual_change_rows.append(
                {
                    "source_file": source_file,
                    "source_text": row["source_text"],
                    "old_svg_text": old_text,
                    "new_svg_text": new_text,
                    "bible_reference_category": row["bible_reference_category"],
                    "translation_action": row["translation_action"],
                    "element_path": row["element_path"],
                    "replacement_details": replacements_made,
                }
            )

    out_path = collapsed_svg_path.with_name(
        f"{collapsed_svg_path.stem}_final_refs_{timestamp}.svg"
    )
    encoding = tree.docinfo.encoding or "UTF-8"
    doctype = tree.docinfo.doctype if tree.docinfo.doctype else None
    tree.write(
        str(out_path),
        encoding=encoding,
        xml_declaration=True,
        pretty_print=False,
        doctype=doctype,
    )

    final_svg_outputs.append(out_path)

    print("source collapsed SVG:", workflow_relative_path(collapsed_svg_path))
    print("output final SVG:", workflow_relative_path(out_path))
    print("reference rows inspected:", inspected)
    print("elements found:", found)
    print("elements changed:", changed)
    print("unresolved paths:", unresolved)
    print("replacement count:", replacement_count)
    print()

df_actual_reference_changes = pd.DataFrame(actual_change_rows)

source collapsed SVG: rev\svg_output_files\StructureOfRevelation_spanish_20260520_2017_collapsed_tspans.svg
output final SVG: rev\svg_output_files\StructureOfRevelation_spanish_20260520_2017_collapsed_tspans_final_refs_20260520_2024.svg
reference rows inspected: 82
elements found: 82
elements changed: 56
unresolved paths: 0
replacement count: 67



In [9]:
for final_svg_path in final_svg_outputs:
    try:
        parser = etree.XMLParser(recover=False, huge_tree=True)
        etree.parse(str(final_svg_path), parser)
        print("XML parse OK:", workflow_relative_path(final_svg_path))
    except etree.XMLSyntaxError as e:
        print("XML parse error:", workflow_relative_path(final_svg_path))
        print(e)

XML parse OK: rev\svg_output_files\StructureOfRevelation_spanish_20260520_2017_collapsed_tspans_final_refs_20260520_2024.svg


In [10]:
review_cols = [
    "source_file",
    "source_text",
    "old_svg_text",
    "new_svg_text",
    "bible_reference_category",
    "translation_action",
    "replacement_details",
]

if df_actual_reference_changes.empty:
    print("No Bible-reference abbreviation changes were made.")
    df_review = pd.DataFrame(columns=review_cols)
else:
    df_review = df_actual_reference_changes[review_cols].copy()

display(df_review)

review_path = JSON_DIR / f"bible_reference_normalization_review_{timestamp}.json"
latest_review_path = JSON_DIR / "bible_reference_normalization_review_latest.json"

review_records = df_review.to_dict(orient="records")
review_path.write_text(json.dumps(review_records, ensure_ascii=False, indent=2), encoding="utf-8")
latest_review_path.write_text(json.dumps(review_records, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved review:", workflow_relative_path(review_path))
print("Saved latest review:", workflow_relative_path(latest_review_path))

,source_file,source_text,old_svg_text,new_svg_text,bible_reference_category,translation_action,replacement_details
0,StructureOfRevelation.svg,"cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6","cf. Ex 7:17-21,Is 49:26,Rev 8:8-10,11:6","cf. Ex 7:17-21,Is 49:26,Ap 8:8-10,11:6",cross_reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."
1,StructureOfRevelation.svg,cf. Rev 14:18,cf. Rev 14:18,cf. Ap 14:18,cross_reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."
2,StructureOfRevelation.svg,"cf. Rev 9:14,13:1","cf. Rev 9:14,13:1","cf. Ap 9:14,13:1",cross_reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."
3,StructureOfRevelation.svg,Rev 16:3,Rev 16:3,Ap 16:3,reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."
4,StructureOfRevelation.svg,Rev 16:4-7,Rev 16:4-7,Ap 16:4-7,reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."
5,StructureOfRevelation.svg,Rev 16:10-11,Rev 16:10-11,Ap 16:10-11,reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."
6,StructureOfRevelation.svg,"cf. Joe 2,Rev 6:12","cf. Joe 2,Rev 6:12","cf. Joe 2,Ap 6:12",cross_reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."
7,StructureOfRevelation.svg,Rev 16:2,Rev 16:2,Ap 16:2,reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."
8,StructureOfRevelation.svg,Rev 16:8-9,Rev 16:8-9,Ap 16:8-9,reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."
9,StructureOfRevelation.svg,Rev 16:12-14,Rev 16:12-14,Ap 16:12-14,reference_only,preserve,"[{'source_alias': 'Rev', 'replacement_alias': ..."


Saved review: rev\json_files\bible_reference_normalization_review_20260520_2024.json
Saved latest review: rev\json_files\bible_reference_normalization_review_latest.json


## Next steps

- Open the final SVGs in a browser for visual inspection.
- Open the final SVGs in Illustrator to verify compatibility.
- If any replacement is too aggressive, adjust `REFERENCE_ALIAS_REPLACEMENTS_ES` or the regex and rerun Notebook 02.
- This notebook intentionally modifies only Bible-reference-related SVG elements identified by the full prepared unit checkpoint.